# Rule discovery over a classifier

`DiscoverySession` is the discovery loop: it admits **new capabilities** into a knowledge ledger only when an independent audit moves a four-number census and the candidate ships with its adversarial twin.

This notebook runs the loop end to end on `breast_cancer`. Each claim is a **classification rule** — "if a row satisfies this condition it is malignant" — and the verdict is decided by a declared precision/support criterion on data the proposer never sees.

* **Proposer** — mines conjunction rules from the training split (and its own validation slice). It never labels.
* **Oracle** — an independent audit on the held-out split. It answers `CERTIFIED / DECLARED / DECLINED / REFUTED`.
* **The laws** — four numbers never a scalar; twins mandatory; tightening free, loosening signed; critical alarms contain the session; retirement keeps its refutation.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # the mjolnir_ml package

import copy
import re
from dataclasses import replace

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from mjolnir_ml import (ApprovalAuthority, Candidate, DiscoveryError,
                        DiscoverySession, certify_admission,
                        verify_admission_certificate)
from mjolnir_ml.discovery import CERTIFIED, DECLARED, DECLINED, REFUTED, _who

data = load_breast_cancer()
X, y, names = data.data, data.target, list(data.feature_names)
F_CONCAVE = names.index("worst concave points")   # the weak single feature
F_RADIUS = names.index("worst radius")            # the second malignant indicator

# Three disjoint roles: mine (proposer), val (the proposer's own sanity check),
# audit (the oracle). The audit split is never touched by the proposer.
X_train, X_audit, y_train, y_audit = train_test_split(
    X, y, test_size=0.45, random_state=0, stratify=y)
X_mine, X_val, y_mine, y_val = train_test_split(
    X_train, y_train, test_size=0.30, random_state=1, stratify=y_train)

print(f"mine {len(y_mine):3d} rows | proposer-validation {len(y_val):3d} | audit {len(y_audit):3d}")
print(f"malignant share (class 0): mine {(y_mine == 0).mean():.3f}  audit {(y_audit == 0).mean():.3f}")

mine 218 rows | proposer-validation  94 | audit 257
malignant share (class 0): mine 0.372  audit 0.374


## The criterion and the claim language

The audit criterion is **declared** before anything is measured, and it is the same for every claim:

| verdict | when |
|---|---|
| `CERTIFIED` | `support >= 20` and `precision >= 0.95` |
| `REFUTED` | `support >= 20` and `precision < 0.60` — a witness exists |
| `DECLINED` | `support < 20`, or precision in the ambiguous band `[0.60, 0.95)` |

A claim is a rule in a small, fixed grammar. `concave` is `worst concave points`, `radius` is `worst radius`; precision is the share of firing audit rows that are malignant.

In [2]:
MIN_SUPPORT, HIGH, LOW = 20, 0.95, 0.60
RULE = re.compile(r"concave >= (?P<t>[0-9.]+)(?: AND radius >= (?P<r>[0-9.]+))?$")

def fmt(t, r=None):
    return f"concave >= {t:.3f}" if r is None else f"concave >= {t:.3f} AND radius >= {r:.2f}"

def fire(claim, Xv, and_as_or=False):
    m = RULE.fullmatch(claim.strip())
    if m is None:
        raise ValueError(f"claim outside the grammar: {claim!r}")
    left = Xv[:, F_CONCAVE] >= float(m["t"])
    if m["r"] is None:
        return left
    right = Xv[:, F_RADIUS] >= float(m["r"])
    return (left | right) if and_as_or else (left & right)   # v0 defect #7

def decide(support, precision):
    if support < MIN_SUPPORT:
        return DECLINED
    if precision >= HIGH:
        return CERTIFIED
    if precision < LOW:
        return REFUTED
    return DECLINED

def evidence(claim, buggy=False):
    m = fire(claim, X_audit, and_as_or=buggy)
    support = int(m.sum())
    precision = float((y_audit[m] == 0).mean()) if support else 0.0
    return support, round(precision, 3)

EVIDENCE = {}

def audit_rule(claim):
    '''The independent oracle: held-out rows only. Returns a verdict and
    records the evidence it used.'''
    support, precision = evidence(claim)
    EVIDENCE[claim] = {"support": support, "precision": precision}
    return decide(support, precision)

def audit_v0_buggy(claim):
    '''The prior audit engine (v0) parsed `AND` as `OR` — defect #7. Its
    record is still in the registry; a new rung must not overturn it silently.'''
    support, precision = evidence(claim, buggy=True)
    return decide(support, precision)

print(fmt(0.080), "->", audit_rule(fmt(0.080)))
print(fmt(0.180), "->", audit_rule(fmt(0.180)))

concave >= 0.080 -> refuted
concave >= 0.180 -> certified


## The registry record

The corpus is the **record of the last full audit**, one verdict per claim, with the decline reason kept for the frontier:

* nine single-feature rules — the fragment v0 could decide;
* eight conjunction rules — **outside v0's fragment**, so v0 declined them (fail-closed quarantine). This is the discovery frontier;
* one declared policy contract — a premise, not a measurement.

The frontier is the decline census **by reason**. A capability is funded by the number of instances it would move, never by a hand-picked example.

In [3]:
GRID_T = [0.02, 0.035, 0.05, 0.065, 0.08, 0.10, 0.12, 0.15, 0.18]
LATTICE = [(0.08, 18.0), (0.10, 18.0), (0.12, 18.0), (0.08, 20.0), (0.10, 20.0),
           (0.02, 12.0), (0.03, 10.0), (0.15, 14.0)]
LEGACY = fmt(0.15, 14.0)           # refuted by the v0 defect
QUARANTINED = {LEGACY}             # a refuted claim needs a human repair request
POLICY = "policy: malignant workup at concave >= 0.150 (declared contract)"

corpus, reasons = {}, {}
for t in GRID_T:
    claim = fmt(t)
    corpus[claim] = audit_rule(claim)
    reasons[claim] = ("precision_ambiguous" if corpus[claim] == DECLINED
                      else "single_feature_audit")
for t, r in LATTICE:
    claim = fmt(t, r)
    corpus[claim] = audit_v0_buggy(claim) if claim == LEGACY else DECLINED
    reasons[claim] = "outside_fragment:conjunction"
corpus[POLICY] = DECLARED
reasons[POLICY] = "policy_contract"

# The signing key lives OUTSIDE the session; the session receives only `verify`.
authority = ApprovalAuthority(b"discovery-demo")
session = DiscoverySession(corpus, reasons=reasons,
                           approval_verifier=authority.verify)

for claim, verdict in corpus.items():
    ev = EVIDENCE.get(claim)
    extra = (f"support={ev['support']:3d} precision={ev['precision']:.3f}"
             if ev else "")
    print(f"{verdict:9s} {claim:56s} {extra}")

st = session.status()
print("\ncensus  :", st["census"])
print("frontier:", st["frontier"])
print("total   :", st["n"], "claims")

refuted   concave >= 0.020                                         support=246 precision=0.390
refuted   concave >= 0.035                                         support=234 precision=0.406
refuted   concave >= 0.050                                         support=223 precision=0.426
refuted   concave >= 0.065                                         support=192 precision=0.495
refuted   concave >= 0.080                                         support=168 precision=0.560
declined  concave >= 0.100                                         support=131 precision=0.710
declined  concave >= 0.120                                         support=110 precision=0.800
declined  concave >= 0.150                                         support= 81 precision=0.938
certified concave >= 0.180                                         support= 48 precision=1.000
declined  concave >= 0.080 AND radius >= 18.00                     
declined  concave >= 0.100 AND radius >= 18.00                     
declined 

## The proposer: mine classification rules from the training split

The proposer is a **rule miner with a declared proposal criterion**, run on data the audit never sees. It mines the conjunction shape `concave >= t AND radius >= r` over a declared threshold lattice, checks each candidate on its own validation slice, and emits:

* `claims` — the rules it proposes as decidable (the census it expects to move);
* `tests` — fresh rules it predicts will certify (a labelled battery, judged by the oracle);
* `twin` — near-miss rules it predicts will stay refuted (the adversarial battery).

A candidate carries **no label for its own claims** — `Candidate()` refuses an `expect` field. Predictions live only in the batteries, where the oracle can falsify them.

In [4]:
def stats(Xv, yv, t, r):
    m = fire(fmt(t, r), Xv)
    s = int(m.sum())
    return s, (float((yv[m] == 0).mean()) if s else 0.0)

def _mine(min_precision, validation_precision, only=None):
    claims, twins = [], []
    for t, r in LATTICE:
        claim = fmt(t, r)
        if claim in QUARANTINED or (only is not None and only.get(claim) != DECLINED):
            continue
        ms, mp = stats(X_mine, y_mine, t, r)
        vs, vp = stats(X_val, y_val, t, r)
        if (ms >= 40 and mp >= min_precision
                and vs >= 18 and vp >= validation_precision):
            claims.append(claim)
        elif vs >= 18 and vp < LOW:
            twins.append((claim, REFUTED))
    tests = []
    for t, r in [(0.09, 21.0), (0.11, 19.0)]:
        ms, mp = stats(X_mine, y_mine, t, r)
        if ms >= 30 and mp >= 0.98:
            tests.append((fmt(t, r), CERTIFIED))
    return Candidate(kind="rung", shape="concave >= t AND radius >= r",
                     claims=tuple(claims), reason="outside_fragment:conjunction",
                     domain="breast_cancer", population=7,
                     tests=tuple(tests), twin=tuple(twins),
                     evidence={"proposal_criterion":
                               f"mine precision>={min_precision}, validation precision>={validation_precision}",
                               "mined_on": "training split only"})

def mine_strict_conjunctions():
    return _mine(0.99, 0.99)

def mine_relaxed_conjunctions():
    return _mine(0.98, 0.95, only=session.corpus)

rung1 = session.propose(mine_strict_conjunctions)[0]
print("proposed by:", rung1.proposed_by)
print("claims     :")
for c in rung1.claims:
    print("   ", c)
print("tests      :", rung1.tests)
print("twin       :", rung1.twin)

proposed by: __main__.mine_strict_conjunctions
claims     :
    concave >= 0.100 AND radius >= 18.00
    concave >= 0.120 AND radius >= 18.00
    concave >= 0.080 AND radius >= 20.00
    concave >= 0.100 AND radius >= 20.00
tests      : (('concave >= 0.090 AND radius >= 21.00', 'certified'), ('concave >= 0.110 AND radius >= 19.00', 'certified'))
twin       : (('concave >= 0.020 AND radius >= 12.00', 'refuted'), ('concave >= 0.030 AND radius >= 10.00', 'refuted'))


## Proposer never disposes

If the auditor were also the proposer, the loop would be certifying its own input. The session refuses **identities**, not intentions: a candidate whose `proposed_by` is the oracle's identity is rejected even if the code is correct.

In [5]:
same_producer = replace(rung1, proposed_by=_who(audit_rule))
try:
    session.evaluate(same_producer, audit_rule)
except PermissionError as ex:
    print("refused:", ex)
print("journaled:", session.journal.payloads("refusal")[-1]["reason"])

refused: self-certification refused: the oracle ('__main__.audit_rule') is the candidate's own proposer — an oracle and its subject must be independent
journaled: self_certification


## Evaluate, then promote

`evaluate` runs the oracle over the candidate's claims and measures the **census** — four numbers, never a score. Promotion is gated on:

| gate | meaning |
|---|---|
| `labelled_tests` | the positive battery is present and the oracle confirms every prediction |
| `twin_holds` | the near-miss battery keeps its verdict |
| `no_new_refuted` | a new `REFUTED` is a defect signal, not a discovery |
| `certified_gained` | the rung must move the census |
| `targeted_decline_moved` | the declared frontier class must shrink |

This is the frontier move — no prior refutation is being overturned — so it is admitted without a signature.

In [6]:
ev1 = session.evaluate(rung1, audit_rule)
for g in ev1.gates:
    print(f"{g.name:24s} {'PASS' if g.passed else 'FAIL'}   "
          f"value={str(g.value):48s} {g.detail[:56]}")
print("\nmay_loosen:", ev1.may_loosen, "| flipped_refuted:", ev1.flipped_refuted,
      "| tightened:", ev1.tightened)
print("census before:", ev1.before)
print("census after :", ev1.after)

adm1 = session.promote(rung1, audit_rule)
print("\nadmitted:", adm1.approved, "-", adm1.reason)
st = session.status()
print("census  :", st["census"])
print("frontier:", st["frontier"])
print("ledger  :", st["ledger"]["active"], "active record(s)")

labelled_tests           PASS   value=2/2                                              a capability earns admission with a labelled test, never
twin_holds               PASS   value=2/2                                              the twin is the battery that fails when the rung is reve
no_new_refuted           PASS   value={'before': 0, 'after': 0}                        a new REFUTED is a defect signal, not a discovery
certified_gained         PASS   value={'before': 0, 'after': 4}                        movement is the point: the rung must move the census
targeted_decline_moved   PASS   value={'before': 4, 'after': 0}                        the declared target is undecided claims of class 'outsid

may_loosen: False | flipped_refuted: 0 | tightened: 0
census before: {'certified': 0, 'declared': 0, 'declined': 4, 'refuted': 0}
census after : {'certified': 4, 'declared': 0, 'declined': 0, 'refuted': 0}

admitted: True - admitted
census  : {'certified': 5, 'declared': 1, 'declined': 6, 

## Loosening is signed

The registry still holds one **legacy** conjunction: `concave >= 0.150 AND radius >= 14.00` was `REFUTED` by audit v0, whose parser read `AND` as `OR` (defect #7). Today's audit says the rule is precise.

That is the dangerous direction — a recorded refutation now being certified. The session refuses to let a new rung resolve the conflict silently: certifying a previously `REFUTED` claim is `may_loosen` and needs an external signature.

In [7]:
print("registry record:", session.corpus[LEGACY])
print("v0 buggy audit :", audit_v0_buggy(LEGACY), "support/precision", evidence(LEGACY, buggy=True))
print("today's audit  :", audit_rule(LEGACY), "support/precision", evidence(LEGACY))

repair = Candidate(kind="legacy_repair", shape="concave >= t AND radius >= r",
                   claims=(LEGACY,), reason="outside_fragment:conjunction",
                   domain="breast_cancer", population=1,
                   proposed_by="registry.legacy-repair-request",
                   tests=((fmt(0.15, 16.0), CERTIFIED),),
                   twin=((fmt(0.12, 14.0), DECLINED),),
                   evidence={"prior_audit": "v0 defect #7 (AND parsed as OR)",
                             "prior_witness_precision": 0.585})
ev2 = session.evaluate(repair, audit_rule)
print("\nmay_loosen:", ev2.may_loosen, "flipped_refuted:", ev2.flipped_refuted)

blocked = session.promote(repair, audit_rule)
print("without approval:", blocked.approved, "-", blocked.reason)

approval = authority.sign(ev2.change_digest)
adm2 = session.promote(repair, audit_rule, approval=approval)
print("with approval   :", adm2.approved, "-", adm2.reason)
print("census          :", session.status()["census"])

registry record: refuted
v0 buggy audit : refuted support/precision (164, 0.585)
today's audit  : certified support/precision (77, 0.974)

may_loosen: True flipped_refuted: 1
without approval: False - approval required: candidate certifies 1 previously REFUTED claim(s)
with approval   : True - admitted
census          : {'certified': 6, 'declared': 1, 'declined': 6, 'refuted': 5}


## A rung that manufactures refutations is contained

The near-miss rules are exactly what an over-general miner would propose. The oracle refutes both, so:

* the promotion gate fails on `no_new_refuted`;
* `tick()` reports a **critical** `new_refuted` alarm;
* `auto_act()` contains the session — admissions are refused until a human lifts it with a signed approval.

Containment is epoch-aware: after the lift, the old evaluation is history and does not re-alarm. (The earlier self-certification refusal is still inside the current epoch, so it reports here too; both criticals clear when containment is lifted.)

In [8]:
bad = Candidate(kind="rung_attempt", shape="concave >= t AND radius >= r",
                claims=(fmt(0.02, 12.0), fmt(0.03, 10.0)),
                proposed_by="notebook.overgeneral-miner",
                reason="outside_fragment:conjunction", domain="breast_cancer",
                population=2,
                tests=((fmt(0.09, 21.0), CERTIFIED),),
                twin=((fmt(0.02, 12.0), REFUTED),),
                evidence={"note": "thresholds lifted from the twin battery"})
bad_adm = session.promote(bad, audit_rule)
print("refused:", bad_adm.reason)

for d in session.tick():
    print(f"{d.name:18s} alarm={d.alarm!s:5s} severity={d.severity:8s} {d.detail}")
print("automatic actions:", [c.reason for c in session.auto_act(session.tick())])

relaxed = session.propose(mine_relaxed_conjunctions)[0]
print("\nwhile contained:", session.promote(relaxed, audit_rule).reason)

try:
    session.uncontain(authority.sign("not-the-containment-digest"))
except PermissionError as ex:
    print("unsigned lift refused:", ex)
session.uncontain(authority.sign(session.containment.change_digest))
print("lifted; tick:", [(d.name, d.alarm) for d in session.tick()])
adm3 = session.promote(relaxed, audit_rule)
print("recovered admission:", adm3.approved, "| claims:", list(relaxed.claims))

refused: gate failed: no_new_refuted, certified_gained, targeted_decline_moved
new_refuted        alarm=True  severity=critical 1 evaluation(s) with after.refuted > before.refuted
self_certification alarm=True  severity=critical 1 self-certification attempt(s)
decline_growth     alarm=False severity=info     0 evaluation(s) with after.declined > before.declined
automatic actions: ['1 evaluation(s) with after.refuted > before.refuted']

while contained: contained:1 evaluation(s) with after.refuted > before.refuted
unsigned lift refused: uncontain requires a valid external approval for sha256:64847e3993cdbfe62ef28f51b1fc389e95b6b06f47a0d8853b522d66972e2731
lifted; tick: [('new_refuted', False), ('self_certification', False), ('decline_growth', False)]
recovered admission: True | claims: ['concave >= 0.080 AND radius >= 18.00']


## The move is certifiable

An admission is only as good as the evidence a third party can re-run. `certify_admission` pins the candidate, both censuses and every gate under an HMAC with a binding id, and it **replays the census-arithmetic gates** (`no_new_refuted`, `certified_gained`) from the published numbers. The oracle-dependent gates (`labelled_tests`, `twin_holds`) are listed as attested — the certificate says what can and cannot be recomputed.

In [9]:
KEY = b"audit-key"
cert = certify_admission(adm1, key=KEY)
print("verified     :", verify_admission_certificate(cert, key=KEY))
print("replayed gates:", cert["replay_gates"])
print("attested gates:", cert["attested_gates"])

tampered = copy.deepcopy(cert)
tampered["after"]["certified"] = 999
try:
    verify_admission_certificate(tampered, key=KEY)
except DiscoveryError as ex:
    print("tampered census refused:", ex)
try:
    verify_admission_certificate(cert, key=b"wrong-key")
except DiscoveryError as ex:
    print("wrong key refused     :", ex)

verified     : True
replayed gates: ['no_new_refuted', 'certified_gained']
attested gates: ['labelled_tests', 'twin_holds', 'targeted_decline_moved']
tampered census refused: signature does not match the certificate
wrong key refused     : signature does not match the certificate


## Union-only retirement

Knowledge is never deleted. Retiring a capability records **why** and reverts its claims to their pre-admission verdicts — shrinking the certified set, which is always allowed without a signature.

In [10]:
payload = session.retire(adm1.candidate_id,
                         "audit v2: a counterexample appeared in a new cohort")
print("reverted claims:", payload["reverted"])
st = session.status()
print("census  :", st["census"])
print("frontier:", st["frontier"])
print("ledger  :", st["ledger"]["active"], "active /",
      st["ledger"]["retired"], "retired")
print("journal verified:", session.journal.verify())

reverted claims: 4
census  : {'certified': 3, 'declared': 1, 'declined': 9, 'refuted': 5}
frontier: {'outside_fragment:conjunction': 6, 'precision_ambiguous': 3}
ledger  : 2 active / 1 retired
journal verified: True


## What the four numbers say

The loop moved the frontier and kept every guard visible:

* `DECLINED -> CERTIFIED` on four frontier claims — the discovery move, admitted only with a passing positive and near-miss battery;
* one `REFUTED -> CERTIFIED` — refused until an external signature, because the registry recorded a refutation;
* two refutations manufactured by an over-general rung — refused, contained, and lifted only with a signature (the epoch keeps the alarm from repeating);
* a retired rung — reverted to `DECLINED`, with the refutation kept in the ledger.

No step produced a score. The census is always four numbers, the twin is always present, and the only word that means "true" is `CERTIFIED` — earned on held-out data by an auditor that never saw the proposer's split.